In [0]:
import os
import urllib.request
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType,
    StringType, TimestampType
)


In [0]:
LANDING = "/Volumes/ifood_case/bronze/landing"
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
MESES = ["2023-01","2023-02","2023-03","2023-04","2023-05"]
 
os.makedirs(LANDING, exist_ok=True)
 
for mes in MESES:
    arquivo = f"yellow_tripdata_{mes}.parquet"
    destino = f"{LANDING}/{arquivo}"
    if os.path.exists(destino):
        print(f"[skip] {arquivo} ja existe")
        continue
    url = f"{BASE_URL}/{arquivo}"
    print(f"[download] {url}")
    urllib.request.urlretrieve(url, destino)
    print(f"[ok] {destino}")
 
print("Arquivos na landing:", os.listdir(LANDING))


[skip] yellow_tripdata_2023-01.parquet ja existe
[skip] yellow_tripdata_2023-02.parquet ja existe
[skip] yellow_tripdata_2023-03.parquet ja existe
[skip] yellow_tripdata_2023-04.parquet ja existe
[skip] yellow_tripdata_2023-05.parquet ja existe
Arquivos na landing: ['yellow_tripdata_2023-01.parquet', 'yellow_tripdata_2023-02.parquet', 'yellow_tripdata_2023-03.parquet', 'yellow_tripdata_2023-04.parquet', 'yellow_tripdata_2023-05.parquet']


In [0]:
schema_yellow = StructType([
    StructField("VendorID",              LongType(),      True),
    StructField("tpep_pickup_datetime",  TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count",       DoubleType(),    True),
    StructField("trip_distance",         DoubleType(),    True),
    StructField("RatecodeID",            DoubleType(),    True),
    StructField("store_and_fwd_flag",    StringType(),    True),
    StructField("PULocationID",          LongType(),      True),
    StructField("DOLocationID",          LongType(),      True),
    StructField("payment_type",          LongType(),      True),
    StructField("fare_amount",           DoubleType(),    True),
    StructField("extra",                 DoubleType(),    True),
    StructField("mta_tax",               DoubleType(),    True),
    StructField("tip_amount",            DoubleType(),    True),
    StructField("tolls_amount",          DoubleType(),    True),
    StructField("improvement_surcharge", DoubleType(),    True),
    StructField("total_amount",          DoubleType(),    True),
    StructField("congestion_surcharge",  DoubleType(),    True),
    StructField("airport_fee",           DoubleType(),    True),
])


In [0]:
# Célula 4 - Leitura com cast explícito (funciona no serverless)
import glob
from functools import reduce

LANDING = "/Volumes/ifood_case/bronze/landing"
TABELA_BRONZE = "ifood_case.bronze.yellow_tripdata_raw"

# Lê arquivo a arquivo e força os tipos com cast explícito
# Resolve o conflito INT/DOUBLE sem precisar de config do Spark
dfs = []
for path in sorted(glob.glob(f"{LANDING}/yellow_tripdata_2023-*.parquet")):
    d = spark.read.parquet(path).select(
        F.col("VendorID").cast("long").alias("VendorID"),
        F.col("tpep_pickup_datetime").cast("timestamp"),
        F.col("tpep_dropoff_datetime").cast("timestamp"),
        F.col("passenger_count").cast("double"),
        F.col("trip_distance").cast("double"),
        F.col("RatecodeID").cast("double"),
        F.col("store_and_fwd_flag").cast("string"),
        F.col("PULocationID").cast("long"),
        F.col("DOLocationID").cast("long"),
        F.col("payment_type").cast("long"),
        F.col("fare_amount").cast("double"),
        F.col("extra").cast("double"),
        F.col("mta_tax").cast("double"),
        F.col("tip_amount").cast("double"),
        F.col("tolls_amount").cast("double"),
        F.col("improvement_surcharge").cast("double"),
        F.col("total_amount").cast("double"),
        F.col("congestion_surcharge").cast("double"),
        F.col("airport_fee").cast("double"),
    )
    dfs.append(d)
    print(f"[lido] {path.split('/')[-1]}: {d.count()} linhas")

df_raw = reduce(lambda a, b: a.unionByName(b), dfs)

# Metadados de linhagem
df_bronze = (
    df_raw
    .withColumn("_ingestao_ts", F.current_timestamp())
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_BRONZE)
)

total = spark.table(TABELA_BRONZE).count()
print(f"\nBronze gravada: {total} linhas")


[lido] yellow_tripdata_2023-01.parquet: 3066766 linhas
[lido] yellow_tripdata_2023-02.parquet: 2913955 linhas
[lido] yellow_tripdata_2023-03.parquet: 3403766 linhas
[lido] yellow_tripdata_2023-04.parquet: 3288250 linhas
[lido] yellow_tripdata_2023-05.parquet: 3513649 linhas

Bronze gravada: 16186386 linhas
